In [1]:
library(tidyverse)
library(ggplot2)
library(lubridate)
library(zoo)
library(stargazer)
library(plm)
library(textreg)

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.2"
Warning message:
"package 'dplyr' was built under R version 4.4.2"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: 'zoo'


The following objects are masked from 'package:base':

    as.Date, as.Date.numeric



Please cite as: 


 Hlavac, Marek (2022). stargazer: Well-Formatted Regression and Summary Statistics Tables.

 R package version 5.2.3. https://CRAN.R-pr

In [2]:
crime_all <- read_csv("Original_Crime_data/crime_in_vancouver.csv")|>
    mutate(date = as.Date(paste(year, month, day, sep = "-")))|>
    filter(!is.na(time_category))|>
    select(type, date, year, month, geo_local_area, time_category)|>
    glimpse()

Rows: 811320 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): type, hundred_block, geo_local_area, time_category
dbl (4): year, month, day, hour

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 811,320
Columns: 6
$ type           <chr> "Break and Enter Commercial", "Break and Enter Commerci…
$ date           <date> 2012-12-14, 2019-03-07, 2019-08-27, 2021-04-26, 2014-0…
$ year           <dbl> 2012, 2019, 2019, 2021, 2014, 2020, 2021, 2022, 2005, 2…
$ month          <dbl> 12, 3, 8, 4, 8, 7, 11, 1, 11, 5, 7, 6, 4, 9, 1, 11, 2, …
$ geo_local_area <chr> "South Cambie", "Fairview", "West End", "West End", "We…
$ time_category  <chr> "day", "night", "night", "night", "night", "night", "ni…


In [3]:
ntl_12_24_monthly <- read_csv("NTL_data/22_ntl_2012_2024_monthly.csv")|>
    filter(!is.na(ntl_mean))|>
    mutate(date = as.Date(paste0(date, "-01"), format = "%Y-%m-%d"))|>
    glimpse()

Rows: 3432 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): name, geo_point_2d, date
dbl (4): ntl_mean, n_pixels, n_non_na_pixels, prop_non_na_pixels

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 3,146
Columns: 7
$ name               <chr> "Downtown", "Hastings-Sunrise", "Kerrisdale", "Marp…
$ geo_point_2d       <chr> "{ \"lon\": -123.11656700827415, \"lat\": 49.280747…
$ ntl_mean           <dbl> 15.101049, 3.424347, 2.195419, 4.155951, 3.277409, …
$ n_pixels           <dbl> 54, 71, 65, 54, 44, 50, 24, 43, 48, 95, 42, 42, 69,…
$ n_non_na_pixels    <dbl> 54, 53, 65, 54, 44, 50, 24, 43, 48, 95, 42, 42, 45,…
$ prop_non_na_pixels <dbl> 1.0000000, 0.7464789, 1.0000000, 1.0000000, 1.00000…
$ date               <date> 2012-01-01, 2012-01-01, 2012-01-01, 2012-01-01, 20…


In [4]:
distinct_crime_types <- crime_all |> distinct(type)
print(distinct_crime_types)

# A tibble: 7 × 1
  type                             
  <chr>                            
1 Break and Enter Commercial       
2 Break and Enter Residential/Other
3 Mischief                         
4 Other Theft                      
5 Theft from Vehicle               
6 Theft of Bicycle                 
7 Theft of Vehicle                 


In [5]:
BEC_crime_type_only <- crime_all |>
    filter(type %in% c("Break and Enter Commercial"))

BEC_crime_trend_all <- BEC_crime_type_only |>
    filter(!is.na(geo_local_area)) |>
    filter(time_category %in% c("night", "day")) |>
    group_by(year, month, geo_local_area, time_category) |>
    summarise(crime_count = n(), .groups = "drop") |>
    mutate(date = as.Date(paste(year, month, "01", sep = "-"))) |>
    pivot_wider(names_from = time_category, values_from = crime_count, 
                names_prefix = "crime_")

BEC_panel_data_ <- BEC_crime_trend_all |>
    left_join(
        ntl_12_24_monthly |>
            select(date, name, ntl_mean),
        by = c("date" = "date", "geo_local_area" = "name")
    ) |>
    filter(!is.na(ntl_mean)) |> 
    mutate(
        log_crime_night = log(crime_night + 1),
        log_crime_day = log(crime_day + 1),
        log_ntl = log(ntl_mean),
        is_downtown = ifelse(geo_local_area == "Downtown", 1, 0)
    )

BEC_panel_data_plm <- pdata.frame(BEC_panel_data_, index = c("geo_local_area", "date"))

BEC_fe_model <- plm(log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
                data = BEC_panel_data_plm, 
                model = "within", 
                effect = "individual")

BEC_panel_data_plm$month_factor <- factor(month(BEC_panel_data_$date))
summary(BEC_fe_model)



Oneway (individual) effect Within Model

Call:
plm(formula = log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
    data = BEC_panel_data_plm, effect = "individual", model = "within")

Unbalanced Panel: n = 22, T = 18-143, N = 1756

Residuals:
      Min.    1st Qu.     Median    3rd Qu.       Max. 
-1.9082529 -0.2586281  0.0019733  0.2941205  1.3434018 

Coefficients:
                     Estimate Std. Error t-value  Pr(>|t|)    
log_ntl             -0.220264   0.063753 -3.4550 0.0005636 ***
log_crime_day        0.262697   0.026391  9.9540 < 2.2e-16 ***
log_ntl:is_downtown  0.192143   0.149463  1.2856 0.1987702    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Total Sum of Squares:    349.82
Residual Sum of Squares: 329.06
R-Squared:      0.05933
Adj. R-Squared: 0.046288
F-statistic: 36.3926 on 3 and 1731 DF, p-value: < 2.22e-16

In [6]:
BER_crime_type_only <- crime_all |>
    filter(type %in% c("Break and Enter Residential/Other"))

BER_crime_trend_all <- BER_crime_type_only |>
    filter(!is.na(geo_local_area)) |>
    filter(time_category %in% c("night", "day")) |>
    group_by(year, month, geo_local_area, time_category) |>
    summarise(crime_count = n(), .groups = "drop") |>
    mutate(date = as.Date(paste(year, month, "01", sep = "-"))) |>
    pivot_wider(names_from = time_category, values_from = crime_count, 
                names_prefix = "crime_")

BER_panel_data_ <- BER_crime_trend_all |>
    left_join(
        ntl_12_24_monthly |>
            select(date, name, ntl_mean),
        by = c("date" = "date", "geo_local_area" = "name")
    ) |>
    filter(!is.na(ntl_mean)) |> 
    mutate(
        log_crime_night = log(crime_night + 1),
        log_crime_day = log(crime_day + 1),
        log_ntl = log(ntl_mean),
        is_downtown = ifelse(geo_local_area == "Downtown", 1, 0)
    )

BER_panel_data_plm <- pdata.frame(BER_panel_data_, index = c("geo_local_area", "date"))

BER_fe_model <- plm(log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
                data = BER_panel_data_plm, 
                model = "within", 
                effect = "individual")

BER_panel_data_plm$month_factor <- factor(month(BER_panel_data_$date))
summary(BER_fe_model)



Oneway (individual) effect Within Model

Call:
plm(formula = log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
    data = BER_panel_data_plm, effect = "individual", model = "within")

Unbalanced Panel: n = 22, T = 91-143, N = 2674

Residuals:
     Min.   1st Qu.    Median   3rd Qu.      Max. 
-1.510572 -0.308571  0.030206  0.332960  1.308533 

Coefficients:
                     Estimate Std. Error t-value  Pr(>|t|)    
log_ntl             -0.255718   0.056216 -4.5489 5.636e-06 ***
log_crime_day        0.348780   0.017561 19.8611 < 2.2e-16 ***
log_ntl:is_downtown  0.628526   0.157223  3.9977 6.572e-05 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Total Sum of Squares:    692.37
Residual Sum of Squares: 594.85
R-Squared:      0.14085
Adj. R-Squared: 0.13307
F-statistic: 144.759 on 3 and 2649 DF, p-value: < 2.22e-16

In [7]:
OT_crime_type_only <- crime_all |>
    filter(type %in% c("Other Theft"))

OT_crime_trend_all <- OT_crime_type_only |>
    filter(!is.na(geo_local_area)) |>
    filter(time_category %in% c("night", "day")) |>
    group_by(year, month, geo_local_area, time_category) |>
    summarise(crime_count = n(), .groups = "drop") |>
    mutate(date = as.Date(paste(year, month, "01", sep = "-"))) |>
    pivot_wider(names_from = time_category, values_from = crime_count, 
                names_prefix = "crime_")

OT_panel_data_ <- OT_crime_trend_all |>
    left_join(
        ntl_12_24_monthly |>
            select(date, name, ntl_mean),
        by = c("date" = "date", "geo_local_area" = "name")
    ) |>
    filter(!is.na(ntl_mean)) |> 
    mutate(
        log_crime_night = log(crime_night + 1),
        log_crime_day = log(crime_day + 1),
        log_ntl = log(ntl_mean),
        is_downtown = ifelse(geo_local_area == "Downtown", 1, 0)
    )

OT_panel_data_plm <- pdata.frame(OT_panel_data_, index = c("geo_local_area", "date"))

OT_fe_model <- plm(log_crime_night ~ log_ntl * is_downtown + log_crime_day,
                data = OT_panel_data_plm, 
                model = "within", 
                effect = "individual")

OT_panel_data_plm$month_factor <- factor(month(OT_panel_data_$date))
summary(OT_fe_model)



Oneway (individual) effect Within Model

Call:
plm(formula = log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
    data = OT_panel_data_plm, effect = "individual", model = "within")

Unbalanced Panel: n = 22, T = 112-143, N = 3008

Residuals:
     Min.   1st Qu.    Median   3rd Qu.      Max. 
-1.454848 -0.238177  0.012824  0.242252  1.310618 

Coefficients:
                     Estimate Std. Error t-value Pr(>|t|)    
log_ntl             -0.094722   0.043323 -2.1864  0.02886 *  
log_crime_day        0.319660   0.017547 18.2171  < 2e-16 ***
log_ntl:is_downtown  0.151514   0.128213  1.1817  0.23740    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Total Sum of Squares:    502.84
Residual Sum of Squares: 452.32
R-Squared:      0.10047
Adj. R-Squared: 0.093232
F-statistic: 111.058 on 3 and 2983 DF, p-value: < 2.22e-16

In [8]:
ToV_crime_type_only <- crime_all |>
    filter(type %in% c("Theft of Vehicle"))

ToV_crime_trend_all <- ToV_crime_type_only |>
    filter(!is.na(geo_local_area)) |>
    filter(time_category %in% c("night", "day")) |>
    group_by(year, month, geo_local_area, time_category) |>
    summarise(crime_count = n(), .groups = "drop") |>
    mutate(date = as.Date(paste(year, month, "01", sep = "-"))) |>
    pivot_wider(names_from = time_category, values_from = crime_count, 
                names_prefix = "crime_")

ToV_panel_data_ <- ToV_crime_trend_all |>
    left_join(
        ntl_12_24_monthly |>
            select(date, name, ntl_mean),
        by = c("date" = "date", "geo_local_area" = "name")
    ) |>
    filter(!is.na(ntl_mean)) |> 
    mutate(
        log_crime_night = log(crime_night + 1),
        log_crime_day = log(crime_day + 1),
        log_ntl = log(ntl_mean),
        is_downtown = ifelse(geo_local_area == "Downtown", 1, 0)
    )

ToV_panel_data_plm <- pdata.frame(ToV_panel_data_, index = c("geo_local_area", "date"))

ToV_fe_model <- plm(log_crime_night ~ log_ntl * is_downtown + log_crime_day,
                data = ToV_panel_data_plm, 
                model = "within", 
                effect = "individual")

ToV_panel_data_plm$month_factor <- factor(month(ToV_panel_data_$date))
summary(ToV_fe_model)


Oneway (individual) effect Within Model

Call:
plm(formula = log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
    data = ToV_panel_data_plm, effect = "individual", model = "within")

Unbalanced Panel: n = 22, T = 14-135, N = 1654

Residuals:
     Min.   1st Qu.    Median   3rd Qu.      Max. 
-1.355070 -0.293735  0.014631  0.315487  1.328340 

Coefficients:
                     Estimate Std. Error t-value  Pr(>|t|)    
log_ntl             -0.264419   0.070068 -3.7738 0.0001666 ***
log_crime_day        0.250871   0.029484  8.5086 < 2.2e-16 ***
log_ntl:is_downtown  0.031000   0.157397  0.1970 0.8438888    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Total Sum of Squares:    340.36
Residual Sum of Squares: 323.34
R-Squared:      0.050008
Adj. R-Squared: 0.036012
F-statistic: 28.5838 on 3 and 1629 DF, p-value: < 2.22e-16

In [9]:
ToB_crime_type_only <- crime_all |>
    filter(type %in% c("Theft of Bicycle"))

ToB_crime_trend_all <- ToB_crime_type_only |>
    filter(!is.na(geo_local_area)) |>
    filter(time_category %in% c("night", "day")) |>
    group_by(year, month, geo_local_area, time_category) |>
    summarise(crime_count = n(), .groups = "drop") |>
    mutate(date = as.Date(paste(year, month, "01", sep = "-"))) |>
    pivot_wider(names_from = time_category, values_from = crime_count, 
                names_prefix = "crime_")

ToB_panel_data_ <- ToB_crime_trend_all |>
    left_join(
        ntl_12_24_monthly |>
            select(date, name, ntl_mean),
        by = c("date" = "date", "geo_local_area" = "name")
    ) |>
    filter(!is.na(ntl_mean)) |> 
    mutate(
        log_crime_night = log(crime_night + 1),
        log_crime_day = log(crime_day + 1),
        log_ntl = log(ntl_mean),
        is_downtown = ifelse(geo_local_area == "Downtown", 1, 0)
    )

ToB_panel_data_plm <- pdata.frame(ToB_panel_data_, index = c("geo_local_area", "date"))

ToB_fe_model <- plm(log_crime_night ~ log_ntl * is_downtown + log_crime_day,
                data = ToB_panel_data_plm, 
                model = "within", 
                effect = "individual")

ToB_panel_data_plm$month_factor <- factor(month(ToB_panel_data_$date))
summary(ToB_fe_model)


Oneway (individual) effect Within Model

Call:
plm(formula = log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
    data = ToB_panel_data_plm, effect = "individual", model = "within")

Unbalanced Panel: n = 22, T = 18-143, N = 1682

Residuals:
      Min.    1st Qu.     Median    3rd Qu.       Max. 
-1.5733638 -0.2588141  0.0015106  0.2796659  1.2815839 

Coefficients:
                     Estimate Std. Error t-value Pr(>|t|)    
log_ntl             -0.036706   0.066887 -0.5488   0.5832    
log_crime_day        0.578792   0.019485 29.7050   <2e-16 ***
log_ntl:is_downtown -0.071073   0.148273 -0.4793   0.6318    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Total Sum of Squares:    462.26
Residual Sum of Squares: 300.95
R-Squared:      0.34897
Adj. R-Squared: 0.33954
F-statistic: 296.062 on 3 and 1657 DF, p-value: < 2.22e-16

In [10]:
TfV_crime_type_only <- crime_all |>
    filter(type %in% c("Theft from Vehicle"))

TfV_crime_trend_all <- TfV_crime_type_only |>
    filter(!is.na(geo_local_area)) |>
    filter(time_category %in% c("night", "day")) |>
    group_by(year, month, geo_local_area, time_category) |>
    summarise(crime_count = n(), .groups = "drop") |>
    mutate(date = as.Date(paste(year, month, "01", sep = "-"))) |>
    pivot_wider(names_from = time_category, values_from = crime_count, 
                names_prefix = "crime_")

TfV_panel_data_ <- TfV_crime_trend_all |>
    left_join(
        ntl_12_24_monthly |>
            select(date, name, ntl_mean),
        by = c("date" = "date", "geo_local_area" = "name")
    ) |>
    filter(!is.na(ntl_mean)) |> 
    mutate(
        log_crime_night = log(crime_night + 1),
        log_crime_day = log(crime_day + 1),
        log_ntl = log(ntl_mean),
        is_downtown = ifelse(geo_local_area == "Downtown", 1, 0)
    )

TfV_panel_data_plm <- pdata.frame(TfV_panel_data_, index = c("geo_local_area", "date"))

TfV_fe_model <- plm(log_crime_night ~ log_ntl * is_downtown + log_crime_day,
                data = TfV_panel_data_plm, 
                model = "within", 
                effect = "individual")

TfV_panel_data_plm$month_factor <- factor(month(TfV_panel_data_$date))
summary(TfV_fe_model)


Oneway (individual) effect Within Model

Call:
plm(formula = log_crime_night ~ log_ntl * is_downtown + log_crime_day, 
    data = TfV_panel_data_plm, effect = "individual", model = "within")

Unbalanced Panel: n = 22, T = 112-143, N = 2992

Residuals:
     Min.   1st Qu.    Median   3rd Qu.      Max. 
-1.928790 -0.250150  0.035549  0.285830  1.415900 

Coefficients:
                     Estimate Std. Error t-value  Pr(>|t|)    
log_ntl             -0.214833   0.046914 -4.5793 4.857e-06 ***
log_crime_day        0.490035   0.016728 29.2949 < 2.2e-16 ***
log_ntl:is_downtown  0.319993   0.138900  2.3038    0.0213 *  
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Total Sum of Squares:    684.44
Residual Sum of Squares: 527.91
R-Squared:      0.22871
Adj. R-Squared: 0.22247
F-statistic: 293.267 on 3 and 2967 DF, p-value: < 2.22e-16

 custom.model.names = c("Break and Enter Commercial", "Break and Enter Residential/Other", "Other Theft", "Theft of Vehicle", "Theft of Bicycle", "Theft from Vehicle"),


In [11]:
# Define custom CSS to adjust table, header, and cell colors
css_style <- "<style>
table, th, td {
  border: 1px solid black; /* Set black borders */
  color: black;            /* Set text color to black */
  background-color: white; /* Set background color to white */
  padding: 5px;
}
caption {
  font-weight: bold;
  margin-bottom: 10px;
}
</style>"

# Generate the HTML summary table using texreg without reordering coefficients
html_output <- htmlreg(
  list(BEC_fe_model, BER_fe_model, OT_fe_model, ToV_fe_model, ToB_fe_model, TfV_fe_model),
  custom.model.names = c("Break and Enter Commercial", "Break and Enter Residential/Other", "Other Theft", "Theft of Vehicle", "Theft of Bicycle", "Theft from Vehicle"),
  caption = "Summary Table of Panel Regression Models",
  single.row = TRUE,      # Print each model's results in a single row
  include.ci = FALSE,     # Omit confidence intervals
  dcolumn = TRUE,         # Improves column alignment
  digits = 3
)

# Prepend the custom CSS style to the HTML output
html_output <- paste0(css_style, "<h1 style='text-align:center;'>Log-Log models for differnt Crime types</h1>", html_output)

# Display the formatted HTML table within the notebook
display_html(html_output)

ERROR: Error in htmlreg(list(BEC_fe_model, BER_fe_model, OT_fe_model, ToV_fe_model, : could not find function "htmlreg"


In [12]:
clustered_bec_se <- sqrt(diag(vcovHC(BEC_fe_model, type = "HC0", cluster = "group")))
clustered_ber_se <- sqrt(diag(vcovHC(BER_fe_model, type = "HC0", cluster = "group")))
clustered_ot_se <- sqrt(diag(vcovHC(OT_fe_model, type = "HC0", cluster = "group")))
clustered_tov_se <- sqrt(diag(vcovHC(ToV_fe_model, type = "HC0", cluster = "group")))
clustered_tob_se <- sqrt(diag(vcovHC(ToB_fe_model, type = "HC0", cluster = "group")))
clustered_tfv_se <- sqrt(diag(vcovHC(TfV_fe_model, type = "HC0", cluster = "group")))

In [14]:
models <- list(
    BEC_fe_model,
    BER_fe_model,
    OT_fe_model,
    ToV_fe_model,
    ToB_fe_model,
    TfV_fe_model
    )

summary_table <- stargazer(models,
    type = "text",                   
    column.labels = c("Break and Enter Commercial", 
                      "Break and Enter Residential/Other", 
                      "Other Theft", "Theft of Vehicle", 
                      "Theft of Bicycle", 
                      "Theft from Vehicle"),
    dep.var.labels = "log(Crime Count)",
    model.numbers = FALSE,
    title = "log-log Hetergonous Regression of Different Crime Types on Average NTL in the Scale of Neighborhoods",
    covariate.labels = c("log(Average NTL)",
                        "Daytime Crime Count",
                        "log(Average NTL) * Downtown"),
    se = list(
        clustered_bec_se,
        clustered_ber_se,
        clustered_ot_se,
        clustered_tov_se,
        clustered_tob_se,
        clustered_tfv_se),
    single.row = FALSE,
    omit.stat = c("f", "ser"),
    notes = "Note: Data Source: VPD OPEN DATA. Clustered by group standard errors appear in parentheses.",
    notes.align = "l",
    star.cutoffs = c(0.1, 0.05, 0.01))


log-log Hetergonous Regression of Different Crime Types on Average NTL in the Scale of Neighborhoods
                                                                                 Dependent variable:                                                     
                            -----------------------------------------------------------------------------------------------------------------------------
                                                                                  log(Crime Count)                                                       
                            Break and Enter Commercial Break and Enter Residential/Other Other Theft Theft of Vehicle Theft of Bicycle Theft from Vehicle
---------------------------------------------------------------------------------------------------------------------------------------------------------
log(Average NTL)                    -0.220***                      -0.256***               -0.095*      -0.264***          -0.03